<!-- NINO26-CABECALHO v1 -->
# 3F — Ondas de Kelvin por SLA/SSH e vento

**Projeto NINO-BRASIL — Oceanografia Física UFPE — Thiago Vilar**  
**Código da fase/letra:** `3F`  ·  **Hipótese:** HIP0

## Descritivo (por que este notebook existe)
Documenta a propagação equatorial oeste-leste compatível com ondas de Kelvin de downwelling — a ponte dinâmica entre rajadas de oeste e o aquecimento do leste.

## Pergunta
A dinâmica equatorial mostra propagação compatível com ondas de Kelvin nos eventos fortes e na formação 2025/26?

## Desafio (hipótese a testar)
A leitura de propagação exige a tensão do vento adequada (não só |u|u zonal); o sinal de Kelvin deve preceder o aquecimento.

## Metodologia (com referências)
Hovmöller de SLA/SSH com setas de propagação e sobreposição da tensão zonal por evento (Bjerknes, 1969; vieses de resposta a WWEs em Cui et al., 2025).

## Contrato de saídas — código predecessor único
Cada figura nasce do **mesmo** `registrar_figura(...)` que congela sua numeric-table sob o **mesmo código**, reescrevendo por **sobreposição** a cada execução:

```python
from nino_brasil.viz import registrar_figura
registrar_figura(fig, "Fig_3F01", fase=3, bloco="F",
                 titulo=..., descricao=..., hipotese="HIP0",
                 notebook="notebooks/fase3/3F_kelvin_sla.ipynb",
                 fontes={"<tabela>": df})   # -> figures/fase3/<codigo>.png + numeric-tables/fase3/<codigo>/
```

| Código | Figura (`figures/fase3/<código>.png`) | Numeric-table (`numeric-tables/fase3/<código>/`) | Descrição |
|---|---|---|---|
| `Fig_3F01` | `Fig_3F01.png` | `Fig_3F01/` | Hovmöller SLA com setas de Kelvin |
| `Fig_3F02` | `Fig_3F02.png` | `Fig_3F02/` | tensão zonal x SLA por evento |

> Padrão em `docs/PADRAO_NOTEBOOKS.md`; validação por `python scripts/validar_figuras.py --strict`.

In [ ]:
import sys; sys.path.insert(0,'.')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import fase3_utils as u

w = pd.read_csv(u.FEAT/'phase3_indices_semanais.csv', parse_dates=['week_ending_sunday']).set_index('week_ending_sunday')
ssh = u.load_ssh_events()
atmo = u.load_atmo()
tau = u.daily_doy_anomaly(u.zonal_wind_stress_proxy(atmo['atm_10m_u_component_of_wind'])).rename('tau_x_anom_nino34_pa')

windows = [
    ('1997-01-01','1998-06-30','1997/98'),
    ('2015-01-01','2016-06-30','2015/16'),
    ('2023-01-01','2024-06-30','2023/24'),
    ('2025-07-01',None,'2025/26 atual'),
]
rows = []
for t0, t1, label in windows:
    seg = ssh.loc[t0:t1] if t1 else ssh.loc[t0:]
    seg = seg.dropna(how='all')
    if seg.empty:
        rows.append({'janela': label, 'inicio': t0, 'fim': t1 or '', 'status': 'sem dados SSH'})
        continue
    lon = seg.columns.astype(float)
    sla = seg - seg.mean(axis=0)
    band = (lon >= 190) & (lon <= 240)
    n34 = sla.loc[:, band].mean(axis=1)
    tau_seg = tau.loc[seg.index.min():seg.index.max()]
    loc = np.nanargmax(sla.values)
    rr, cc = np.unravel_index(loc, sla.values.shape)
    rows.append({
        'janela': label,
        'inicio': str(seg.index.min().date()),
        'fim': str(seg.index.max().date()),
        'max_sla_m': round(float(np.nanmax(sla.values)), 3),
        'min_sla_m': round(float(np.nanmin(sla.values)), 3),
        'data_max_sla': str(seg.index[rr].date()),
        'longitude_max_sla': u.lon_label(float(lon[cc])),
        'sla_medio_nino34_m': round(float(n34.mean()), 3),
        'tau_x_anom_media_pa': round(float(tau_seg.mean()), 5) if not tau_seg.empty else np.nan,
        'frac_tau_x_oeste': round(float((tau_seg > 0).mean()), 3) if not tau_seg.empty else np.nan,
        'leitura': 'SLA positivo e tau_x medio positivo favorecem leitura de downwelling/Kelvin' if (not tau_seg.empty and float(n34.mean()) > 0 and float(tau_seg.mean()) > 0) else 'leitura diagnostica; verificar inclinacao no Hovmoller',
    })
resumo = pd.DataFrame(rows)
u.save_table(resumo, 'phase3F_kelvin_eventos_resumo.csv', index=False)
print(resumo.to_string(index=False))


In [ ]:
import matplotlib.dates as mdates
import matplotlib.patheffects as pe

def kelvin_pulses(sla, lon, n=2):
    """Detecta os n maiores pulsos de SLA no Pacifico oeste (150E-180) para
    ancorar setas esquematicas de propagacao Kelvin (~2.4 m/s => ~65 dias
    para cruzar 100 graus de longitude)."""
    west = sla.loc[:, (lon >= 150) & (lon <= 180)].mean(axis=1).rolling(15, min_periods=5).mean().dropna()
    pulses = []
    s = west.copy()
    for _ in range(n):
        if s.empty or float(s.max()) < 0.02:
            break
        t = s.idxmax()
        pulses.append((t, float(s.max())))
        s = s[(s.index < t - pd.Timedelta(days=75)) | (s.index > t + pd.Timedelta(days=75))]
    return pulses

fig, axes = plt.subplots(len(windows), 1, figsize=(12.8, 13.2), sharex=True)
fig.subplots_adjust(hspace=0.10)
last_pc = None
seta_rows = []
for k, (ax, (t0, t1, title)) in enumerate(zip(axes, windows)):
    seg = ssh.loc[t0:t1] if t1 else ssh.loc[t0:]
    seg = seg.dropna(how='all')
    if seg.empty:
        ax.text(.5, .5, 'sem dados SSH nesta janela', transform=ax.transAxes, ha='center')
        continue
    lon = seg.columns.values.astype(float)
    sla = seg - seg.mean(axis=0)
    last_pc = ax.pcolormesh(lon, seg.index, sla.values, cmap='RdYlBu_r', vmin=-0.20, vmax=0.20, shading='auto')
    u.add_nino34_lon_band(ax, label=(k == 0))
    u.format_lon_axis(ax, xlabel='Longitude oficial (120E -> 80W; oeste para leste)' if k == len(windows)-1 else '')
    ax.set_ylabel('Data')
    ax.text(0.008, 0.97, title, transform=ax.transAxes, ha='left', va='top', fontsize=10, weight='bold',
            bbox={'boxstyle':'round,pad=0.25','facecolor':'white','edgecolor':'#555','alpha':.9})
    ax.tick_params(labelsize=7)
    ax.yaxis.set_major_locator(mdates.MonthLocator(interval=2))
    ax.yaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    # SETAS GRANDES: propagacao Kelvin oeste->leste (~2.4 m/s)
    for pi, (tp, amp) in enumerate(kelvin_pulses(sla, lon)):
        t_end = tp + pd.Timedelta(days=65)
        if t_end > seg.index.max():
            t_end = seg.index.max()
        dias = (t_end - tp).days
        if dias < 25:
            continue
        ax.annotate('', xy=(255, mdates.date2num(t_end)), xytext=(163, mdates.date2num(tp)),
                    arrowprops={'arrowstyle': '-|>,head_width=0.55,head_length=1.1', 'lw': 3.2,
                                'color': '#111827', 'shrinkA': 0, 'shrinkB': 0,
                                'path_effects': [pe.withStroke(linewidth=5.5, foreground='white')]})
        if pi == 0:
            ax.annotate('onda de Kelvin\n(oeste -> leste, ~2.4 m/s)', xy=(209, mdates.date2num(tp + (t_end-tp)/2)),
                        xytext=(0, 9), textcoords='offset points', ha='center', va='bottom', fontsize=8, weight='bold',
                        color='#111827', path_effects=[pe.withStroke(linewidth=3, foreground='white')])
        seta_rows.append({'janela': title, 'pulso_oeste_em': str(tp.date()), 'sla_oeste_m': round(amp, 3),
                          'chegada_estimada_leste': str(t_end.date()),
                          'velocidade_assumida_m_s': 2.4,
                          'leitura': 'pulso de SLA no Pacifico oeste propagando para leste (downwelling Kelvin)'})
fig.suptitle('3F - Hovmoller SLA/SSH: setas = propagacao equatorial de ondas de Kelvin (oeste -> leste)', y=0.995)
if last_pc is not None:
    fig.colorbar(last_pc, ax=axes, label='SLA local (m); neutro em amarelo', shrink=.75)
u.add_note(axes[-1], 'Faixas vermelhas inclinadas para a direita = agua alta indo de oeste para leste = Kelvin de downwelling.\nSLA local = SSH menos a media da janela por longitude; faixa cinza = Nino 3.4.', loc='lower right')
setas = pd.DataFrame(seta_rows)
u.save_table(setas, 'phase3F_kelvin_setas.csv', index=False)
u.stamp_caption(fig, variavel='SLA local por longitude; setas de propagacao Kelvin', area='Pacifico equatorial; Nino 3.4 sombreado', periodo='janelas 1997/98, 2015/16, 2023/24, 2025/26', fonte='GLORYS12/GLO12 SSH', extra='setas ancoradas nos maiores pulsos de SLA 150E-180 (rolagem 15 dias); velocidade esquematica 2.4 m/s; tabela phase3F_kelvin_setas.csv')
u.save_fig(fig, '3F1_hovmoller_sla_kelvin.png')
print(setas.to_string(index=False))
plt.show()


In [ ]:
fig, axes = plt.subplots(len(windows), 1, figsize=(12.8, 8.8), sharex=False)
for ax, (t0, t1, title) in zip(axes, windows):
    seg = ssh.loc[t0:t1] if t1 else ssh.loc[t0:]
    seg = seg.dropna(how='all')
    if seg.empty:
        ax.text(.5, .5, 'sem dados', transform=ax.transAxes, ha='center')
        continue
    lon = seg.columns.astype(float)
    sla = seg - seg.mean(axis=0)
    n34 = sla.loc[:, (lon >= 190) & (lon <= 240)].mean(axis=1).rolling(14, min_periods=4).mean()
    taux = tau.loc[seg.index.min():seg.index.max()].rolling(14, min_periods=4).mean()
    ax.plot(n34.index, n34, color='#1d4ed8', lw=1.4, label='SLA medio Nino 3.4')
    ax.axhline(0, color='0.4', lw=.6)
    ax.set_ylabel('SLA (m)', color='#1d4ed8')
    ax2 = ax.twinx()
    ax2.plot(taux.index, taux, color='#7c2d12', lw=1.0, alpha=.85, label='tau_x anom.')
    ax2.axhline(0, color='#7c2d12', lw=.5, alpha=.5)
    ax2.set_ylabel('tau_x anom. (Pa)', color='#7c2d12')
    ax.set_title(title, fontsize=9)
    ax.tick_params(labelsize=7); ax2.tick_params(labelsize=7)
axes[0].legend(loc='upper left', fontsize=8)
fig.suptitle('3F - Coerencia entre SLA medio na Nino 3.4 e anomalia de estresse zonal')
u.add_note(axes[-1], 'tau_x positivo = anomalia de oeste no proxy local;\nSLA positivo indica expansao/recarga da coluna d\'agua.', loc='lower right')
u.stamp_caption(fig, variavel='SLA medio Nino 3.4 + tau_x_anom_nino34_pa', area='Nino 3.4; Pacifico equatorial', periodo='janelas de eventos fortes/super e 2025/26', fonte='GLORYS12/GLO12 SSH, ERA5')
u.save_fig(fig, '3F2_taux_sla_eventos.png')
plt.show()


**Leitura do 3F.** A etapa 3F fica restrita a dinamica equatorial: SLA/SSH mostra a resposta da coluna d'agua e `tau_x_anom_nino34_pa` mostra o acoplamento vento-superficie. As setas grandes do 3F1 marcam explicitamente a propagacao Kelvin: cada seta parte do maior pulso de SLA no Pacifico oeste (150E-180) e cruza ate ~105W com velocidade esquematica de 2.4 m/s (~65 dias); os pulsos e datas estao em `phase3F_kelvin_setas.csv`. Faixas vermelhas inclinadas para a direita acompanhando as setas confirmam a leitura de downwelling. A leitura e diagnostica e qualitativa, nao um detector automatico formal de ondas.


<!-- NINO26-REFERENCIAS v1 -->
## Referências Bibliográficas

1. Bjerknes, J. (1969). Atmospheric Teleconnections from the Equatorial Pacific. *Monthly Weather Review*, 97(3), 163-172. https://doi.org/10.1175/1520-0493(1969)097<0163:ATFTEP>2.3.CO;2
2. Cui, J., et al. (2025). Mixed Layer and Oceanic Kelvin Wave Response to Equatorial Pacific WWEs. *JGR-Oceans*. https://doi.org/10.1029/2025JC023275

Relação completa em `Artigos_Referências/Referências_Bibliográficas.xls`.